# LANGCHAIN + OLLAMA

In [ ]:
#https://python.langchain.com/docs/tutorials/rag/

## Data Indexing

#### Using RecursiveUrlLoader

##### Adding extractor: To parse HTML into a more human/LLM-friendly format

In [ ]:
import requests

import re
from bs4 import BeautifulSoup
from langchain.document_loaders.recursive_url_loader import RecursiveUrlLoader
from langchain.document_loaders import PyPDFLoader
from langchain.utils.html import PREFIXES_TO_IGNORE_REGEX, SUFFIXES_TO_IGNORE_REGEX



url = "https://licensespring.com/blog/glossary/open-source-software/"

# List of all URLs to process
urls = [
    "https://licensespring.com/blog/glossary/open-source-software/",
    "https://opensource.guide/maintaining-balance-for-open-source-maintainers/",
    "https://opensource.guide/how-to-contribute/",
    "https://opensource.guide/starting-a-project/",
    "https://opensource.guide/finding-users/",
    "https://opensource.guide/building-community/",
    "https://opensource.guide/best-practices/",
    "https://opensource.guide/leadership-and-governance/",
    "https://opensource.guide/getting-paid/",
    "https://opensource.guide/code-of-conduct/",
    "https://opensource.guide/metrics/",
    "https://opensource.guide/legal/"
  
]

HTML_documents = []

for url in urls:
    headers = {'User-Agent': 'Mozilla/5.0'}  # Helps prevent blocking by some servers
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        document = BeautifulSoup(response.content, "html5lib")
        HTML_documents.extend(document)
        # Now you can work with 'soup' to extract or manipulate the HTML content
    else:
        print(f"Failed to retrieve the page. Status code: {response.status_code}")

PDF_documents = []

pdf_loader1 = PyPDFLoader(file_path="OPEN SOURCE SOFTWARE GUIDELINES.pdf")
pdf_loader2 = PyPDFLoader(file_path="Creating Impactful.pdf")

pdf_data1 = pdf_loader1.load()
pdf_data2 = pdf_loader2.load()

# Add PDF documents to all_documents
PDF_documents.extend(pdf_data1)
PDF_documents.extend(pdf_data2)

### Split documents

To handle lengthy text efficiently, the Langchain text splitter divides text into smaller, semantically meaningful units and combines them into larger chunks with defined size and overlap. Here, I used the RecursiveCharacterTextSplitter to process scraped documents into manageable chunks while preserving context continuity.

`chunk_size` and `chunk_overlap` effects to the prompt size

execeed promt size causes error `prompt size exceeds the context window size and cannot be processed`

**HTMLHeaderTextSplitter**

It is a "structure-aware" text splitter that splits text at the HTML element level and adds metadata for each header "relevant" to any given chunk. It can return chunks element by element or combine elements with the same metadata, with the objectives of (a) keeping related text grouped (more or less) semantically and (b) preserving context-rich information encoded in document structures. It can be used with other text splitters as part of a chunking pipeline.

In [ ]:
# print(HTML_documents)

In [ ]:
# PDF_documents

In [ ]:
from langchain_text_splitters import HTMLHeaderTextSplitter 

# source: https://python.langchain.com/docs/how_to/split_html/ 

# Define headers to split on 
headers_to_split_on = [ 
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
    ("h4", "Header 4"),
    ("h5", "Header 5"),
    ("h6", "Header 6"),
] 

# Initialize the HTMLHeaderTextSplitter 
HTML_splitter = HTMLHeaderTextSplitter(headers_to_split_on) 


chunked_HTML_documents = [HTML_splitter.split_text(str(html)) for html in HTML_documents]

chunked_HTML_documents = [doc for sublist in chunked_HTML_documents for doc in sublist] 
chunked_HTML_documents 

**Recursively split by character** 

This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is ["\n\n", "\n", " ", ""]. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.
* How the text is split: by list of characters.
* 
How the chunk size is measured: by number of characters

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

#source: https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/recursive_text_splitter/

#chunk size: maximum number of characters each chunk can contain
#chunk overlap: the number of overlapping characters between consecutive chunks, ensure context continuity between chunks
PDF_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200) #Splits each document into chunks
chunked_PDF_documents = PDF_splitter.split_documents(PDF_documents)
print(chunked_PDF_documents)

In [ ]:
print(chunked_PDF_documents[:1000])

###  Create Vector Embedding

After splitting the text, it is converted into vector embeddings using machine learning models like HuggingFace's all-MiniLM-L6-v2. These high-dimensional vectors capture semantic meanings and enable efficient operations such as grouping, searching, and measuring sentence similarity based on semantic closeness, surpassing traditional keyword-based methods.

#### configuration

In [ ]:
import os

# Set environment variables directly in the notebook
os.environ['INIT_INDEX'] = 'true'
os.environ['INDEX_PERSIST_DIRECTORY'] = './data2/chromadb'
os.environ['TARGET_URL'] = 'https://open5gs.org/open5gs/docs/'
os.environ['HTTP_PORT'] = '7654'
os.environ['MONGO_HOST'] = 'localhost'
os.environ['MONGO_PORT'] = '27017'
os.environ['MONGO_USER'] = 'testuser'
os.environ['MONGO_PASS'] = 'testpass'


In [ ]:

# define init index
INIT_INDEX = os.getenv('INIT_INDEX', 'false').lower() == 'true'

# vector index persist directory
INDEX_PERSIST_DIRECTORY = os.getenv('INDEX_PERSIST_DIRECTORY', "./data2/chromadb")

# target url to scrape
TARGET_URL =  os.getenv('TARGET_URL', "https://open5gs.org/open5gs/docs/")

# http api port
HTTP_PORT = os.getenv('HTTP_PORT', 7654)

# mongodb config host, username, password
MONGO_HOST = os.getenv('MONGO_HOST', 'localhost')
MONGO_PORT = os.getenv('MONGO_PORT', 27017)
MONGO_USER = os.getenv('MONGO_USER', 'testuser')
MONGO_PASS = os.getenv('MONGO_PASS', 'testpass')

**all-MiniLM-L6-v2**

This is a sentence-transformers model: It maps sentences & paragraphs to a 384 dimensional dense vector space and can be used for tasks like clustering or semantic search.

In [ ]:
# !pip install --upgrade transformers

In [ ]:
# from langchain.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

# https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
#https://www.sbert.net/docs/sentence_transformer/pretrained_models.html

# Initialize embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L12-v2")


### Store Vector Embedding in Chroma

Chroma (ChromaDB) is an open-source vector database that stores embeddings and their metadata, enabling efficient semantic search by processing text-based data semantically, unlike traditional databases. It enhances the system's ability to quickly retrieve and compare relevant information, improving the accuracy of responses to user queries.

In [ ]:
chunked_documents= chunked_HTML_documents + chunked_PDF_documents
# chunked_documents

In [ ]:
from langchain.vectorstores import Chroma


# Helper function to split data into batches
def batch_data(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

# Split documents into smaller batches
BATCH_SIZE = 166  # Maximum allowed batch size
for batch in batch_data(chunked_documents, BATCH_SIZE):
    # Add each batch to the Chroma vector store
    vectordb = Chroma.from_documents(
        documents=batch,
        embedding=embeddings,
        persist_directory=INDEX_PERSIST_DIRECTORY
    )


In [ ]:
# Check if the vector store contains any documents
print(f"Number of documents in the vector store: {len(vectordb)}")

In [ ]:
from langchain_chroma import Chroma
# load index
vectordb = Chroma(persist_directory=INDEX_PERSIST_DIRECTORY,embedding_function=embeddings)

## Retrieval and generation

The system offers an API that allows users to ask questions related to Open5GS documentation, with user sessions identified by a user_id for tracking. The API is designed for ease of use, enabling intuitive interactions, and in real-world scenarios, user identification could be managed via an Authorization header.

### Create Vector Embedding of Question

When a user submits a question through the API, the system converts it into a vector embedding, which is automatically generated by the ConversationalRetrievalChain, enabling semantic search of relevant documents in the vector database.

**Qwen2.5-14B.

* Type: Causal Language Models
  
Training Stage: Pretraining 
*Architecture: transformers with RoPE, SwiGLU, RMSNorm, and Attention QKV bias.
* Number of Parameters: 14.7B
* Number of Paramaters (Non-Embedding): 13.1BB
Number of Lay: 48er
* Number of Attention Heads (GQA): 40 for Q and 8 forv.K* V
Context Length: Full 072768 tokens

In [ ]:
from langchain_community.llms import Ollama
from langchain.llms import OpenAI 
# llama2 llm which runs with ollama 
# ollama expose an api for the llam in `localhost:11434` 
llm = Ollama(
    # model="llama3.2",
    model="qwen2.5:14b",
    base_url="http://localhost:11434",
    verbose=True, #provide detailed logs, messages, or output about what it's doing,
    temperature=0
) 

ConversationalRetrievalChain Deprecated since version 0.1.17
#https://api.python.langchain.com/en/latest/chains/langchain.chains.conversational_retrieval.base.ConversationalRetrievalChain.html
#https://python.langchain.com/api_reference/langchain/chains/langchain.chains.conversational_retrieval.base.ConversationalRetrievalChain.html

**MultiQueryRetriever**: 

#https://python.langchain.com/docs/how_to/MultiQueryRetriever/

In [ ]:
from typing import List

from langchain_core.output_parsers import BaseOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field


# Output parser will split the LLM result into a list of queries
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Output parser for a list of lines."""

    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        return list(filter(None, lines))  # Remove empty lines


output_parser = LineListOutputParser()

QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI language model assistant. Your task is to generate five 
    different versions of the given user question to retrieve relevant documents from a vector 
    database. By generating multiple perspectives on the user question, your goal is to help
    the user overcome some of the limitations of the distance-based similarity search. 
    Provide these alternative questions separated by newlines.
    Original question: {question}""",
)

# Chain
llm_chain = QUERY_PROMPT | llm | output_parser

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever

# Create conversation with the correct method (invoke instead of __call__)
# retriever_from_llm = MultiQueryRetriever.from_llm(
#     retriever=vectordb.as_retriever(),  # Convert the vector database to a retriever
#     llm=llm, 
#     parser_key="lines"
#     # "lines" is the key (attribute name) of the parsed output
# )

retriever = MultiQueryRetriever(
    retriever=vectordb.as_retriever(), 
    llm_chain=llm_chain, 
    parser_key="lines"
)  # "lines" is the key (attribute name) of the parsed output


In [ ]:
# Set logging for the queries
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)


In [ ]:
# Ask the first question
question = "Give me the definition of OSS?"

docs = retriever.invoke(question)
len(docs) 

In [ ]:
print(retriever.llm_chain)

In [ ]:
docs

In [ ]:
# Repeat the process for subsequent questions
question_2 = "What is early determination of distribution policy?"

unique_docs2 = retriever_from_llm.invoke(question_2)
len(unique_docs2) 

In [ ]:
unique_docs2

In [ ]:
# Repeat the process for subsequent questions
question_3 = "What are the key characteristics of Open-source software?"

# unique_docs3 = retriever_from_llm.invoke(question_3)
# len(unique_docs3) 

In [ ]:
# unique_docs3

In [ ]:
# Repeat the process for subsequent questions
question_4 = "why a person or organization would want to open source a project?"

# unique_docs4 = retriever_from_llm.invoke(question_4)
# len(unique_docs4) 

In [ ]:
# unique_docs4

In [ ]:
print(llm) 

In [ ]:
#changer le modele à quen2.5:14b
#choix model de embedding
#quelle justifie technique de RAG: langchain 'prompts particuliers'
#utiliser autre parser
#utilisr d'autres methodes pour preporcessing
#decoupage en chunks: par ponctuation, titres...


#ajouter template de question

# ConversationalRetrievalChain!!! A changer MultiQueryRetreiver + explain why

## Evaluation

### Document relevancy

In [ ]:
!pip install langchain-groq

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq

# Data model
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""

    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


# LLM with function call
# llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0,api_key= "") changed
llm = ChatGroq(model="qwen2.5:14b", temperature=0, api_key="") 
structured_llm_grader = llm.with_structured_output(GradeDocuments)

# Prompt
system = """You are a grader assessing relevance of a retrieved document to a user question. \n 
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""
grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

retrieval_grader = grade_prompt | structured_llm_grader

In [ ]:
llm

In [ ]:
retrieval_grader

### Filter out the non-relevant docs

In [ ]:
for doc in docs:
    print(f"Page: {doc.metadata['page']}\n\nSource: {doc.metadata['source']}\n\nContent: {doc.page_content}\n")

In [ ]:
docs_to_use = []
for doc in docs:
    print(doc.page_content, '\n', '-'*50)
    res = retrieval_grader.invoke({"question": question, "document": doc.page_content})
    print(res,'\n')
    if res.binary_score == 'yes':
        docs_to_use.append(doc)

In [ ]:
docs_to_use

### Generate results

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Prompt
system = """You are an assistant for question-answering tasks. Answer the question based upon your knowledge. 
Use three-to-five sentences maximum and keep the answer concise."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved documents: \n\n <docs>{documents}</docs> \n\n User question: <question>{question}</question>"),
    ]
)

# LLM
llm = ChatGroq(model="qwen2.5:14b", temperature=0, api_key="") 

# Post-processing
def format_docs(docs):
    return "\n".join(f"<doc{i+1}>:\nPage:{doc.metadata['page']}\nSource:{doc.metadata['source']}\nContent:{doc.page_content}\n</doc{i+1}>\n" for i, doc in enumerate(docs))

# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
generation = rag_chain.invoke({"documents":format_docs(docs_to_use), "question": question})
print(generation)

### Hallucinations

In [ ]:
# Data model
class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in 'generation' answer."""

    binary_score: str = Field(
        ...,
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )

# LLM with function call
llm = ChatGroq(model="qwen2.5:14b", temperature=0, api_key="") 

structured_llm_grader = llm.with_structured_output(GradeHallucinations)

# Prompt
system = """You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. \n 
    Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts."""
hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Set of facts: \n\n <facts>{documents}</facts> \n\n LLM generation: <generation>{generation}</generation>"),
    ]
)

hallucination_grader = hallucination_prompt | structured_llm_grader

response = hallucination_grader.invoke({"documents": format_docs(docs_to_use), "generation": generation})
print(response)

**Highlight used documents**

In [ ]:
from typing import List
from langchain.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

# Data model
class HighlightDocuments(BaseModel):
    """Return the specific part of a document used for answering the question."""

    id: List[str] = Field(
        ...,
        description="List of id of docs used to answers the question"
    )

    page: List[str] = Field(
        ...,
        description="List of pages used to answers the question"
    )

    source: List[str] = Field(
        ...,
        description="List of sources used to answers the question"
    )

    segment: List[str] = Field(
        ...,
        description="List of direct segements from used documents that answers the question"
    )

# LLM
# llm = ChatGroq(model="mixtral-8x7b-32768", temperature=0, api_key= "") changed
llm = ChatGroq(model="qwen2.5:14b", temperature=0, api_key="") 

# parser
parser = PydanticOutputParser(pydantic_object=HighlightDocuments)

# Prompt
system = """You are an advanced assistant for document search and retrieval. You are provided with the following:
1. A question.
2. A generated answer based on the question.
3. A set of documents that were referenced in generating the answer.

Your task is to identify and extract the exact inline segments from the provided documents that directly correspond to the content used to 
generate the given answer. The extracted segments must be verbatim snippets from the documents, ensuring a word-for-word match with the text 
in the provided documents.

Ensure that:
- (Important) Each segment is an exact match to a part of the document and is fully contained within the document text.
- The relevance of each segment to the generated answer is clear and directly supports the answer provided.
- (Important) If you didn't used the specific document don't mention it.

Used documents: <docs>{documents}</docs> \n\n User question: <question>{question}</question> \n\n Generated answer: <answer>{generation}</answer>

<format_instruction>
{format_instructions}
</format_instruction>
"""


prompt = PromptTemplate(
    template= system,
    input_variables=["documents", "question", "generation"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Chain
doc_lookup = prompt | llm | parser

# Run
lookup_response = doc_lookup.invoke({"documents":format_docs(docs_to_use), "question": question, "generation": generation})

In [ ]:
for id, page, source, segment in zip(lookup_response.id, lookup_response.page, lookup_response.source, lookup_response.segment):
    print(f"ID: {id}\nPage: {page}\nSource: {source}\nText Segment: {segment}\n")